# Brasileirão Predictor AI — Análise Exploratória de Dados ⚽📊

Este notebook demonstra o carregamento das partidas históricas do Brasileirão Série A (OpenFootball CC0), a distribuição de gols por rodada e a evolução dos ratings Elo calculados cronologicamente.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.database import SessionLocal, init_db
from src.schemas.models import Match
from src.serving.predictor import MatchPredictor

init_db()
db = SessionLocal()
print("Conectado ao banco de dados com sucesso!")

In [ ]:
# Carregando partidas em DataFrame
query = db.query(Match.match_id, Match.season, Match.round, Match.goals_home, Match.goals_away, Match.status)
df_matches = pd.DataFrame(query.all(), columns=["match_id", "season", "round", "goals_home", "goals_away", "status"])
print(f"Total de partidas carregadas: {len(df_matches)}")
df_matches.head()

In [ ]:
# Distribuição de resultados (Vitória Mandante / Empate / Vitória Visitante)
def get_outcome(row):
    if row["goals_home"] > row["goals_away"]:
        return "Vitória Mandante"
    elif row["goals_home"] == row["goals_away"]:
        return "Empate"
    else:
        return "Vitória Visitante"

df_matches["resultado"] = df_matches.apply(get_outcome, axis=1)
df_matches["resultado"].value_counts(normalize=True).round(3) * 100

In [ ]:
# Testando inferência unificada via MatchPredictor
predictor = MatchPredictor()
pred = predictor.predict_by_match_id(1, model_type="numeric_lgbm")
print("Previsão da Partida 1:")
print(pred)